In [9]:
import numpy as np
import math
import random

In [8]:
class Mathlib:
    def randn(*shape):
        pass

    def clip(val, minVal=0, maxVal=1):
        return(min(maxVal, max(minVal, val)))

    def mean(vals):
        return(sum(vals)/len(vals))

    def dot2Vectors(vector1, vector2):
        out = 0
        for v1, v2 in zip(vector1, vector2):
            out += v1*v2
        return(out)

    def normalize(values):
        out = []
        total = sum(values)
        for v in values:
            out.append(v/total)
        return(out)

In [ ]:
class Activation:
    def RecLE(val):
        return(max(0, val))    #< by clipping 0, you de-linearize your data

    def none(val):
        return(val)

    def softmax(val):
        return(math.e**val)    #< useful to ensure no negative values

    def protectedSoftmax(vals):
        out = []
        maxVal = max(vals)
        for v in vals:
            out.append(Activation.softmax(v-maxVal))   #< softmax, but between 0 and 1
        return(out)


In [6]:
class Loss:
    def calcLoss(vals, index):
        return(-math.log(Mathlib.clip(vals[index], 1e-7, 1-1e-7)))

In [ ]:
class Neuron:
    """ A single neuron, stores its inputs, weights, bias, and activation function """
    def __init__(self, inputs, weights, bias, activationFunc=Activation.none):
        self.inputs = inputs      #< all outputs from previous layer
        self.weights = weights    #< one weight for every input
        self.bias = bias          #< one bias per neuron
        self.activationFunc = activationFunc

    def calc(self):
        """
        Compute the neuron's output.
        Multiply each input by its corresponding weight, sum the
        results, add the bias and then apply the activation function.
        
        multiply the inputs [i1, i2.. in] with weights [w1, w1... wn] to get i1*w1+i2*w2... +in*wn, then add the bias
        """
        
        return(self.activationFunc(Mathlib.dot2Vectors(self.inputs, self.weights)+self.bias))


In [ ]:
class Layer:
    """A collection of neurons forming a single layer in the network"""
    def __init__(self, inputs, weights, biases, normalize=True, layerActivation=Activation.none, *args, **kwargs):
        self.out = None
        self.normalize = normalize
        self.layerActivation = layerActivation  #< layer activation function
        self.neurons = self._createNeurons(inputs, weights, biases, *args, **kwargs)  # create neuron objects

    def _createNeurons(self, inputs, weights, biases, *args, **kwargs):
        neurons = []
        for w, b in zip(weights, biases):
            neurons.append(Neuron(inputs, w, b, *args, **kwargs))
        return(neurons)
    
    def calc(self):
        """Calculate outputs for all neurons in the layer, apply layer-level
        activation and optional normalization."""
        self.out = [neuron.calc() for neuron in self.neurons]
        self.out = self.layerActivation(self.out)      #< Apply layer activation function (e.g., softmax, protectedSoftmax)
        if self.normalize:
            self.out = Mathlib.normalize(self.out)     #< convert output from 0 to 1
        return(self.out)

    def calcLoss(self, correctIndex):
        """Convenience wrapper for computing loss for a single sample."""
        return(Loss.calcLoss(self.out, correctIndex))


In [ ]:
class Batch:
    """ Process multiple samples at once in a batch."""
    def __init__(self, inputsBatch, *args, **kwargs):
        self.batch = self._createBatch(inputsBatch, *args, **kwargs)  # create a Layer for each sample

    def _createBatch(self, inputsBatch, *args, **kwargs):
        """Helper method to create a Layer object for each sample in the batch."""
        batch = []
        for inputs in inputsBatch:
            batch.append(Layer(inputs, *args, **kwargs))  # instantiate Layer and add to batch
        return(batch)
    
    def calc(self):
        """Run all layers in the batch and return their outputs."""
        return([layer.calc() for layer in self.batch])
    
    def calcLoss(self, correctIndexes):
        """Compute mean loss across the batch given a list of target indexes."""
        return(Mathlib.mean([self.batch[i].calcLoss(correctIndexes[i]) for i in range(len(self.batch))]))  # average of per-sample losses


In [ ]:
class Train:
    """ TBD """
    def __init__(self, weights, biases, learningRate=0.01):
        self.weights = weights
        self.biases = biases
        self.learningRate = learningRate

    def step(self, inputs, correctIndex, activation=Activation.protectedSoftmax):
        """Perform a single gradient descent update for one example.

        Args:
            inputs: list of input features for the sample
            correctIndex: index of the true class in the output vector
            activation: activation used at layer level (default softmax variant)
        """
        layer = Layer(inputs, self.weights, self.biases, layerActivation=activation)
        out = layer.calc()
        for i in range(len(self.weights)) :
            # target probability is 1 for correct class, 0 otherwise (eg: 0 for cat, 0 for dog, 1 for horse)
            target = 0
            if i == correctIndex:
                target = 1
            error = out[i] - target
            for j in range(len(self.weights[i])):
                self.weights[i][j] -= self.learningRate * error * inputs[j]
            self.biases[i] -= self.learningRate * error

    def train(self, inputsBatch, correctIndexes, epochs=1, activation=Activation.protectedSoftmax):
        """Train for a number of epochs """
        for _ in range(epochs):
            for inputs, correct in zip(inputsBatch, correctIndexes):
                self.step(inputs, correct, activation)

In [12]:

np.random.seed(0)

def createSetData():
    inputs = [[1.0, 2.0, 3.0, 2.5],
              [2.0, 5.0, -1.0, 2.0],
              [-1.5, 2.7, 3.3, -0.8]]
    weights = np.array([[0.2, 0.8, -0.5, 1.0],
               [0.5, -0.91, 0.26, -0.5],
               [-0.26, -0.27, 0.17, 0.87]])

    biases = [2.0, 3.0, 0.5]
    print("expected output:")
    print(np.dot(inputs, weights.T) + biases)
    print("-------------------------\n\n\n")
    return(inputs, weights, biases)

def createData(n_inputs, n_neurons, batchSize=1):
    inputs = np.random.randn(batchSize, n_neurons)
    weights = 0.01*np.random.randn(n_inputs, n_neurons)
    biases = np.zeros(n_neurons)
    return(inputs, weights, biases)

inputs, weights, biases = createSetData()

batch = Batch(inputs, weights, biases, layerActivation=Activation.protectedSoftmax)
# batch = Batch(inputs, weights, biases, activationFunc=Activation.softmax, layerActivation=Activation.none)
out = batch.calc()
print(np.array(out))

print(Loss.calcLoss([0.7, 0.2, 0.1], 0))
print(Loss.calcLoss([0.5, 0.1, 0.4], 1))
print(Loss.calcLoss([0.02, 0.9, 0.08], 1))

print(batch.calcLoss([0,0,0]))

expected output:
[[ 4.8    1.21   2.385]
 [ 8.9   -1.81   0.2  ]
 [ 1.41   1.051  0.026]]
-------------------------



[[8.95282664e-01 2.47083068e-02 8.00090293e-02]
 [9.99811129e-01 2.23163963e-05 1.66554348e-04]
 [5.13097164e-01 3.58333899e-01 1.28568936e-01]]
0.35667494393873245
2.3025850929940455
0.10536051565782628
0.25936490708526483
